# Day 4

## 用代码做分词（Tokenizing）

In [ ]:
# 导入 tiktoken：OpenAI 的分词器（tokenizer），把文字切成 token（模型计费/上下文的最小单位）
import tiktoken

# 按模型名取对应编码表（encoding）；不同模型的 tokenizer 可能不同
encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

# encode：把字符串切成 token id 列表；计费与上下文窗口都按 token 计
tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [ ]:
tokens

In [ ]:
# 遍历（for 循环）：逐个处理序列里的每一项
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

In [ ]:
encoding.decode([326])

# 另一个主题！

### 「记忆」的错觉

你们很多人可能已经知道了。但对不知道的人来说——这可能是一个「顿悟」时刻！

In [ ]:
# 导入标准库 os（操作系统相关，用来读环境变量 Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv

# 加载 .env 文件：把 API Key 等密钥读入进程环境（override=True 表示覆盖已有同名变量）
load_dotenv(override=True)
# 用 os.getenv 读取环境变量里的密钥；找不到时返回 None
api_key = os.getenv('OPENAI_API_KEY')

# 条件判断：若密钥/结果为空，则提示用户检查配置
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

### 你应该对下一个单元格在做什么非常熟悉！

_我正在创建 OpenAI Python 客户端库的新实例，这是一个轻量封装，用于向端点发起 HTTP 调用，以调用 GPT LLM 或其他 LLM 提供商_

In [ ]:
# 从 openai 导入 OpenAI 客户端类：用它调用 Chat Completions 等 API（Application Programming Interface）
from openai import OpenAI

# 创建 OpenAI 客户端；不传参时默认读环境变量里的 API Key
openai = OpenAI()

### 发给 OpenAI 的消息是一个字典列表

In [ ]:
# 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [ ]:
# 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

### 好，现在让我们问一个后续问题

In [ ]:
# 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [ ]:
# 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

### 等等，什么？？

我们刚才明明告诉你了！

怎么回事？？

关键在于：每次对 LLM 的调用都是完全无状态（STATELESS）的。每一次都是全新的调用。作为 AI 工程师，**我们的工作**就是设计各种技术，让 LLM 看起来像有「记忆」。

In [ ]:
# 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [ ]:
# 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

## 总结回顾

如果这对你来说显而易见，抱歉——但加强记忆仍然有益：

1. 每次对 LLM 的调用都是无状态的
2. 我们每次都在输入提示中传入迄今为止的完整对话
3. 这造成了 LLM 有记忆的错觉——它似乎保持了对话上下文
4. 但这是一个技巧；是每次提供完整对话所带来的副产品
5. LLM 只是预测序列中最可能的下一个 token；如果该序列包含 "My name is Ed"，稍后又有 "What's my name?"，那么它会预测…… Ed！

ChatGPT 产品正是使用了这个技巧——每次你发送消息时，传入的都是完整对话。

「这是否意味着我们每次都要为迄今为止的全部对话额外付费」

当然如此。而这正是我们**想要**的。我们希望 LLM 在回顾整个对话的基础上预测序列中的下一个 token。我们希望发生这样的计算，所以需要为它支付电费！
